# Локальное использование LLM на macOS M2

Этот ноутбук посвящён локальному запуску моделей на **Apple Silicon (M2, unified memory)** и показывает несколько практических путей работы с одной и той же семейством моделей.

Что мы делаем:
- оцениваем, какие модели реально помещаются в память ноутбука;
- запускаем `Qwen3-4B` через `Transformers + MPS`;
- запускаем **квантованную** модель через **MLX 4-bit**;
- запускаем `Qwen3-8B` в **thinking mode** через **MLX 4-bit**;
- запускаем GGUF-модель через **llama.cpp + Metal**;
- показываем локальный API-доступ через **Ollama + OpenAI-compatible API**.


## Импорты и утилиты

Задача этой секции:
- импортировать библиотеки для всех дальнейших сценариев;
- проверить состояние окружения и доступность `MPS`, `MLX`, `llama.cpp` и `Ollama`;
- подготовить общие вспомогательные функции для замера времени, оценки числа токенов и очистки памяти.


In [1]:
from __future__ import annotations

import gc
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

results: list[list[str]] = []
STOP_OLLAMA_AFTER_RUN = os.getenv("STOP_OLLAMA_AFTER_RUN", "0").strip().lower() not in ("0", "false", "no", "")
MODELS = {
    "hf_4b": "Qwen/Qwen3-4B",
    "mlx_4b_q": "Qwen/Qwen3-4B-MLX-4bit",
    "mlx_8b_q": "Qwen/Qwen3-8B-MLX-4bit",
    "gguf_repo": "Qwen/Qwen3-4B-GGUF",
    "gguf_file": "Qwen3-4B-Q4_K_M.gguf",
    "ollama_model": os.getenv("OLLAMA_MODEL", "hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M"),
}

try:
    from tabulate import tabulate
except Exception:
    tabulate = None

try:
    import torch
except Exception as exc:
    torch = None
    TORCH_IMPORT_ERROR = exc
else:
    TORCH_IMPORT_ERROR = None

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
except Exception as exc:
    AutoModelForCausalLM = None
    AutoTokenizer = None
    TRANSFORMERS_IMPORT_ERROR = exc
else:
    TRANSFORMERS_IMPORT_ERROR = None

try:
    from huggingface_hub import hf_hub_download
except Exception as exc:
    hf_hub_download = None
    HF_IMPORT_ERROR = exc
else:
    HF_IMPORT_ERROR = None

try:
    import mlx.core as mx
except Exception as exc:
    mx = None
    MLX_CORE_IMPORT_ERROR = exc
else:
    MLX_CORE_IMPORT_ERROR = None

try:
    from mlx_lm import generate as mlx_generate
    from mlx_lm import load as mlx_load
    from mlx_lm.sample_utils import make_sampler
except Exception as exc:
    mlx_generate = None
    mlx_load = None
    make_sampler = None
    MLX_IMPORT_ERROR = exc
else:
    MLX_IMPORT_ERROR = None

try:
    from llama_cpp import Llama
except Exception as exc:
    Llama = None
    LLAMA_CPP_IMPORT_ERROR = exc
else:
    LLAMA_CPP_IMPORT_ERROR = None

try:
    from openai import OpenAI
except Exception as exc:
    OpenAI = None
    OPENAI_IMPORT_ERROR = exc
else:
    OPENAI_IMPORT_ERROR = None


def shell(cmd: list[str]) -> str:
    try:
        return subprocess.check_output(cmd, text=True).strip()
    except Exception:
        return "N/A"


def tool_path(name: str) -> str:
    local_bin = Path(sys.executable).parent / name
    if local_bin.exists():
        return str(local_bin)
    return shutil.which(name) or "not found"


def resolve_ollama_gguf_path(model_name: str) -> Path | None:
    ollama_bin = tool_path("ollama")
    if ollama_bin == "not found":
        return None
    try:
        modelfile = subprocess.check_output([ollama_bin, "show", "--modelfile", model_name], text=True)
    except Exception:
        return None
    for line in modelfile.splitlines():
        if line.startswith("FROM /"):
            candidate = Path(line.split("FROM ", 1)[1].strip())
            if candidate.exists():
                return candidate
    return None


def total_memory_gb() -> float:
    raw = shell(["sysctl", "-n", "hw.memsize"])
    try:
        return int(raw) / 1024**3
    except Exception:
        return float("nan")


def pretty_seconds(value: float) -> str:
    return f"{value:.2f}s"


def estimate_tokens(text: str, tokenizer=None) -> int:
    if tokenizer is not None:
        try:
            return len(tokenizer.encode(text))
        except Exception:
            pass
    return max(1, int(len(text.split()) * 1.3))


def record_result(name: str, backend: str, duration_s: float, tokens: int, note: str = "") -> None:
    tps = tokens / duration_s if duration_s > 0 else 0.0
    row = [
        name,
        backend,
        pretty_seconds(duration_s),
        str(tokens),
        f"{tps:.2f}",
        note,
    ]
    for idx, existing in enumerate(results):
        if existing[0] == name:
            results[idx] = row
            break
    else:
        results.append(row)


def reset_results() -> None:
    results.clear()


def clear_memory() -> None:
    gc.collect()
    if torch is not None:
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
        try:
            if hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
                torch.mps.empty_cache()
        except Exception:
            pass
    if mx is not None:
        try:
            if hasattr(mx, "synchronize"):
                mx.synchronize()
        except Exception:
            pass
        try:
            if hasattr(mx, "clear_cache"):
                mx.clear_cache()
        except Exception:
            pass


def mps_status() -> str:
    if torch is None:
        return f"torch import failed: {TORCH_IMPORT_ERROR}"
    try:
        built = torch.backends.mps.is_built()
        available = torch.backends.mps.is_available()
        return f"built={built}, available={available}"
    except Exception as exc:
        return f"MPS check failed: {exc}"


reset_results()

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Machine:", platform.machine())
print(f"Unified memory: {total_memory_gb():.1f} GB")
print("MPS:", mps_status())
print("docker:", tool_path("docker"))
print("ollama:", tool_path("ollama"))
print("jupyter:", tool_path("jupyter"))
print("cmake:", tool_path("cmake"))
print("mlx_lm:", "OK" if mlx_load is not None else f"missing ({MLX_IMPORT_ERROR})")
print("mlx core:", "OK" if mx is not None else f"missing ({MLX_CORE_IMPORT_ERROR})")
print("transformers:", "OK" if AutoTokenizer is not None else f"missing ({TRANSFORMERS_IMPORT_ERROR})")
print("llama_cpp:", "OK" if Llama is not None else f"missing ({LLAMA_CPP_IMPORT_ERROR})")
print("openai sdk:", "OK" if OpenAI is not None else f"missing ({OPENAI_IMPORT_ERROR})")
print("results:", "reset to empty table")
print("ollama auto-stop:", STOP_OLLAMA_AFTER_RUN)


Python: 3.12.7
Platform: macOS-15.6-arm64-arm-64bit
Machine: arm64
Unified memory: 16.0 GB
MPS: built=True, available=True
docker: /usr/local/bin/docker
ollama: /usr/local/bin/ollama
jupyter: /Users/newuser/Documents/repo/llm-app/.venv/bin/jupyter
cmake: /Users/newuser/Documents/repo/llm-app/.venv/bin/cmake
mlx_lm: OK
mlx core: OK
transformers: OK
llama_cpp: OK
openai sdk: OK
results: reset to empty table
ollama auto-stop: True


## Математика памяти для M2

Задача этой ячейки - прикинуть, какие сценарии для `4B` и `8B` моделей выглядят реалистично на ноутбуке с unified memory.

Здесь мы не делаем точный профайлинг, а получаем практическую оценку: хватит ли памяти на веса модели, KV-cache и запас под систему.


In [2]:
def estimate_memory_gb(params_b: float, bytes_per_param: float, context_tokens: int = 8192) -> float:
    model_gb = params_b * bytes_per_param
    kv_cache_gb = (context_tokens / 1000) * 0.10
    return model_gb + kv_cache_gb

cases = [
    ("Qwen3-4B fp16/float16", 4.0, 2.0),
    ("Qwen3-4B q4", 4.0, 0.55),
    ("Qwen3-8B fp16/float16", 8.0, 2.0),
    ("Qwen3-8B q4", 8.0, 0.55),
]

rows = []
ram_gb = total_memory_gb()
for name, params_b, bpp in cases:
    total_gb = estimate_memory_gb(params_b, bpp)
    verdict = "OK" if total_gb < ram_gb * 0.75 else "tight"
    rows.append([name, f"~{total_gb:.2f} GB", verdict])

if tabulate is not None:
    print(tabulate(rows, headers=["Model", "Estimated working set", "M2 verdict"], tablefmt="github"))
else:
    for row in rows:
        print(" | ".join(row))

print("\nНаблюдение: по оценке памяти квантованные варианты требуют заметно меньше ресурсов, чем full-precision запуск.")


| Model                 | Estimated working set   | M2 verdict   |
|-----------------------|-------------------------|--------------|
| Qwen3-4B fp16/float16 | ~8.82 GB                | OK           |
| Qwen3-4B q4           | ~3.02 GB                | OK           |
| Qwen3-8B fp16/float16 | ~16.82 GB               | tight        |
| Qwen3-8B q4           | ~5.22 GB                | OK           |

Наблюдение: по оценке памяти квантованные варианты требуют заметно меньше ресурсов, чем full-precision запуск.


## Qwen3-4B в более полном формате через Transformers + MPS

Задача этой ячейки - показать локальный запуск модели через стандартный стек `Transformers`, используя `MPS` как основной ускоритель на Mac.

Если `MPS` в текущем окружении недоступен, ячейка корректно пропустится.


In [3]:
clear_memory()

if torch is None or AutoModelForCausalLM is None or AutoTokenizer is None:
    print("⏭ Пропуск: не хватает torch/transformers")
elif not torch.backends.mps.is_available():
    print("⏭ Пропуск: MPS сейчас недоступен в этом окружении")
else:
    device = torch.device("mps")
    model_id = MODELS["hf_4b"]
    prompt = "Explain entropy in simple terms and give one real-world example."

    print(f"🏋️ Loading {model_id} on MPS in float16...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
    model.to(device)

    inputs = tokenizer(text, return_tensors="pt").to(device)
    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            max_new_tokens=160,
        )
    duration = time.time() - start

    completion_ids = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(completion_ids, skip_special_tokens=True)
    tokens = len(completion_ids)

    print(answer[:1500])
    print(f"\n⏱ Duration: {duration:.2f}s | Tokens: {tokens}")
    record_result("Qwen3-4B (float16)", "Transformers + MPS", duration, tokens, "higher precision path")
    print("Наблюдение: эта ячейка использует стандартный стек Transformers + MPS и обычно является одной из самых тяжёлых по времени и памяти.")

    del model, tokenizer, inputs, output
    clear_memory()


🏋️ Loading Qwen/Qwen3-4B on MPS in float16...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

**Entropy**, in simple terms, is a measure of **disorder** or **randomness** in a system. The higher the entropy, the more disorganized or chaotic the system is.

Think of it like a room: if the room is tidy, with things in their places, that’s low entropy. But if the room is messy, with clothes on the floor, books scattered, and toys everywhere, that’s high entropy.

**Real-world example**:  
Imagine a cup of hot coffee left out on a table. Over time, the heat from the coffee spreads out into the surrounding air. The coffee cools down, and the heat energy becomes more spread out in the environment. This process increases the overall **disorder** of the system (the coffee and the air), which means

⏱ Duration: 29.28s | Tokens: 160
Наблюдение: эта ячейка использует стандартный стек Transformers + MPS и обычно является одной из самых тяжёлых по времени и памяти.


## Qwen3-4B квантованная через MLX 4-bit

Задача этой ячейки - показать основной практический сценарий локального запуска квантованной модели на Apple Silicon через `MLX`.


In [4]:
clear_memory()

if mlx_load is None:
    print("⏭ Пропуск: mlx-lm не установлен")
else:
    model_id = MODELS["mlx_4b_q"]
    print(f"🤏 Loading {model_id}...")

    model, tokenizer = mlx_load(model_id)
    messages = [{"role": "user", "content": "Explain entropy in simple terms and keep the answer under 6 bullet points. /no_think"}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    sampler = make_sampler(temp=0.7, top_p=0.8) if make_sampler is not None else None

    start = time.time()
    answer = mlx_generate(
        model,
        tokenizer,
        prompt=prompt,
        sampler=sampler,
        verbose=False,
        max_tokens=220,
    )
    duration = time.time() - start
    tokens = estimate_tokens(answer, tokenizer)

    print(answer[:1500])
    print(f"\n⏱ Duration: {duration:.2f}s | Estimated output tokens: {tokens}")
    record_result("Qwen3-4B (4-bit)", "MLX", duration, tokens, "quantized on Apple Silicon")
    print("Наблюдение: эта ячейка показывает квантованный запуск через MLX и обычно даёт короткое время ответа при умеренном расходе памяти.")

    del model, tokenizer
    clear_memory()


🤏 Loading Qwen/Qwen3-4B-MLX-4bit...


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

- Entropy is a measure of disorder or randomness in a system.  
- It tells us how much energy is unavailable for doing work.  
- Over time, systems tend to move toward higher entropy, or more disorder.  
- Higher entropy means more possible arrangements of particles.  
- It’s a key concept in understanding the second law of thermodynamics.

⏱ Duration: 2.66s | Estimated output tokens: 72
Наблюдение: эта ячейка показывает квантованный запуск через MLX и обычно даёт короткое время ответа при умеренном расходе памяти.


## Qwen3-8B в thinking mode через MLX 4-bit

Задача этой ячейки - показать, что на `M2` можно локально запустить более тяжёлую `8B` модель в режиме reasoning/thinking и получить осмысленный ответ.


In [5]:
clear_memory()

if mlx_load is None:
    print("⏭ Пропуск: mlx-lm не установлен")
else:
    model_id = MODELS["mlx_8b_q"]
    print(f"🧠 Loading {model_id} with thinking enabled...")

    model, tokenizer = mlx_load(model_id)
    messages = [{"role": "user", "content": "How many r's are in the word 'strawberry'?"}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    sampler = make_sampler(temp=0.6, top_p=0.95) if make_sampler is not None else None

    start = time.time()
    answer = mlx_generate(
        model,
        tokenizer,
        prompt=prompt,
        sampler=sampler,
        verbose=False,
        max_tokens=320,
    )
    duration = time.time() - start
    tokens = estimate_tokens(answer, tokenizer)

    print(answer[:2000])
    print(f"\n⏱ Duration: {duration:.2f}s | Estimated output tokens: {tokens}")
    record_result("Qwen3-8B (thinking)", "MLX 4-bit", duration, tokens, "Qwen thinking mode on M2")
    print("Наблюдение: эта ячейка показывает более тяжёлый reasoning-сценарий через MLX 4-bit, поэтому время ответа обычно выше, чем у 4B-модели.")

    del model, tokenizer
    clear_memory()


🧠 Loading Qwen/Qwen3-8B-MLX-4bit with thinking enabled...


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

<think>
Okay, let's see. The user is asking how many times the letter 'r' appears in the word 'strawberry'. Hmm, first I need to make sure I have the word spelled correctly. Let me write it out: S-T-R-A-W-B-E-R-R-Y. Wait, is that right? Let me check again. Strawberry. S-T-R-A-W-B-E-R-R-Y. Yeah, that's the correct spelling. Now, I need to count the number of 'r's in there.

Let me go through each letter one by one. Starting with the first letter: S. Not an 'r'. Next is T. Still not. Then comes R. Okay, that's one 'r'. Then A, W, B, E. So far, only one 'r'. Now, the next letters: R again. That's the second 'r'. Then another R. Wait, so after the first R, there's another R later on. Let me count again to be sure. Let's break it down:

S - not an 'r'
T - not
R - 1st 'r'
A - no
W - no
B - no
E - no
R - 2nd 'r'
R - 3rd 'r'
Y - no

Wait, hold on. Let me check again. The word is spelled S-T-R-A-W-B-E-R-R-Y. So after the first R, there's another R after E, and then another R before Y? Wait, no.

## GGUF + llama.cpp (Metal)

Задача этой ячейки - показать локальный запуск GGUF-модели через `llama.cpp` и `Metal`, то есть через отдельный desktop-friendly рантайм.

Если подходящий GGUF уже лежит внутри локальной `Ollama`, ноутбук попробует переиспользовать его и не будет тянуть ещё одну большую копию модели.


In [6]:
clear_memory()

if Llama is None or hf_hub_download is None or AutoTokenizer is None:
    print("⏭ Пропуск: нужен llama-cpp-python, huggingface-hub и transformers")
else:
    repo_id = MODELS["gguf_repo"]
    filename = MODELS["gguf_file"]
    local_dir = Path("models")
    local_dir.mkdir(exist_ok=True)
    model_path = local_dir / filename

    if model_path.exists():
        print(f"📦 Reusing local GGUF: {model_path}")
    else:
        ollama_gguf = resolve_ollama_gguf_path(MODELS["ollama_model"])
        if ollama_gguf is not None:
            model_path = ollama_gguf
            print(f"📦 Reusing GGUF from Ollama storage: {model_path}")
        else:
            print(f"📥 Downloading {filename} from {repo_id}...")
            model_path = Path(hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(local_dir)))

    chat_tokenizer = AutoTokenizer.from_pretrained(MODELS["hf_4b"])
    messages = [{"role": "user", "content": "Explain why Makefiles still matter in modern engineering. /no_think"}]
    prompt = chat_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    llm = Llama(
        model_path=str(model_path),
        n_gpu_layers=-1,
        n_ctx=4096,
        verbose=False,
    )

    start = time.time()
    output = llm(
        prompt,
        max_tokens=220,
        temperature=0.7,
        top_p=0.8,
        echo=False,
    )
    duration = time.time() - start

    answer = output["choices"][0]["text"]
    tokens = estimate_tokens(answer, chat_tokenizer)

    print(answer[:1500])
    print(f"\n⏱ Duration: {duration:.2f}s | Estimated output tokens: {tokens}")
    record_result("Qwen3-4B (Q4_K_M)", "llama.cpp + Metal", duration, tokens, filename)
    print("Наблюдение: эта ячейка использует GGUF-модель через llama.cpp + Metal и позволяет сравнить отдельный рантайм с MLX и Transformers.")

    llm.close()
    del llm, chat_tokenizer, output
    clear_memory()


📦 Reusing GGUF from Ollama storage: /Users/newuser/.ollama/models/blobs/sha256-7485fe6f11af29433bc51cab58009521f205840f5b4ae3a32fa7f92e8534fdf5


llama_context: n_ctx_per_seq (4096) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)


ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)


ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)


...

ggml_metal_init: skipping kernel_flash_attn_ext_vec_bf16_hk576_hv512 (not supported)


ggml_metal_init: skipping kernel_cpy_f32_bf16                      (not supported)


ggml_metal_init: skipping kernel_cpy_bf16_f32                      (not supported)


ggml_metal_init: skipping kernel_cpy_bf16_bf16                     (not supported)


Makefiles are still highly relevant in modern engineering, despite the rise of more advanced tools and automation in software development. Here's a breakdown of why they remain important:

---

### **1. Portability and Cross-Platform Compatibility**
- **Uniform Build Process**: Makefiles define a **consistent build process** that works across different operating systems (Linux, macOS, Windows) and architectures (x86, ARM, etc.).
- **No Dependency on Build Tools**: Unlike CMake or other modern tools, Makefiles don't require specific build environments. They are **language-agnostic** and can be used with any compiler or build system.

---

### **2. Control and Precision**
- **Fine-Grained Control**: Makefiles allow developers to **define exact build steps**, including compilation, linking, and cleaning. This level of control is essential for **complex projects** (e.g., embedded systems, large C++ applications).
- **Conditional Logic**: Makefiles support **conditional statements** and **t

## Ollama + OpenAI-compatible API

Задача этой ячейки - показать локальный запуск модели через `Ollama` и обращение к ней как к привычному `OpenAI-compatible API`.

Перед запуском ячейки подними Ollama и один раз скачай модель, например:

```bash
ollama pull hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M
```


In [7]:
clear_memory()

if OpenAI is None:
    print("⏭ Пропуск: openai sdk не установлен")
else:
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    model_name = MODELS["ollama_model"]

    try:
        print(f"🌐 Querying Ollama model: {model_name}")
        start = time.time()
        resp = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "user", "content": "Reply with one short sentence about Unix pipelines."}
            ],
            temperature=0.2,
            max_tokens=800,
        )
    except Exception as exc:
        print(f"⏭ Пропуск: Ollama API недоступен на http://localhost:11434 ({exc})")
    else:
        duration = time.time() - start
        answer = (resp.choices[0].message.content or "").strip()
        tokens = estimate_tokens(answer)

        print(answer[:1500])
        print(f"\n⏱ Duration: {duration:.2f}s | Estimated output tokens: {tokens}")
        record_result("Qwen3 via Ollama", "OpenAI-compatible API", duration, tokens, model_name)
        print("Наблюдение: эта ячейка обращается к локальной модели через OpenAI-compatible API и показывает формат работы через отдельный сервис.")
        if STOP_OLLAMA_AFTER_RUN:
            ollama_bin = shutil.which("ollama")
            if ollama_bin:
                stop_result = subprocess.run([ollama_bin, "stop", model_name], capture_output=True, text=True)
                if stop_result.returncode == 0:
                    print("Ollama: модель остановлена после ячейки, чтобы освободить память.")
                else:
                    message = (stop_result.stderr or stop_result.stdout or "unknown error").strip()
                    print(f"Ollama: не удалось автоматически остановить модель ({message}).")
            else:
                print("Ollama: автостоп включён, но CLI не найден на хосте. Если сервер работает в Docker, останови модель или контейнер вручную.")
    finally:
        try:
            client.close()
        except Exception:
            pass


🌐 Querying Ollama model: hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M


Unix pipelines link commands via pipes, enabling sequential data processing.

⏱ Duration: 10.85s | Estimated output tokens: 13
Наблюдение: эта ячейка обращается к локальной модели через OpenAI-compatible API и показывает формат работы через отдельный сервис.


Ollama: модель остановлена после ячейки, чтобы освободить память.


## Итоговая таблица


In [8]:
if not results:
    print("Пока нет результатов: либо окружение не готово, либо секции были пропущены.")
else:
    headers = ["Scenario", "Backend", "Duration", "Output tokens", "Tokens/sec", "Notes"]
    if tabulate is not None:
        print(tabulate(results, headers=headers, tablefmt="github"))
    else:
        print(headers)
        for row in results:
            print(row)
    fastest = max(results, key=lambda row: float(row[4]))
    print(f"\nНаблюдение: в этом прогоне максимальная скорость генерации получилась у сценария '{fastest[0]}' через '{fastest[1]}'.")


| Scenario            | Backend               | Duration   |   Output tokens |   Tokens/sec | Notes                           |
|---------------------|-----------------------|------------|-----------------|--------------|---------------------------------|
| Qwen3-4B (float16)  | Transformers + MPS    | 29.28s     |             160 |         5.46 | higher precision path           |
| Qwen3-4B (4-bit)    | MLX                   | 2.66s      |              72 |        27.05 | quantized on Apple Silicon      |
| Qwen3-8B (thinking) | MLX 4-bit             | 17.61s     |             320 |        18.18 | Qwen thinking mode on M2        |
| Qwen3-4B (Q4_K_M)   | llama.cpp + Metal     | 9.47s      |             220 |        23.24 | Qwen3-4B-Q4_K_M.gguf            |
| Qwen3 via Ollama    | OpenAI-compatible API | 10.85s     |              13 |         1.2  | hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M |

Наблюдение: в этом прогоне максимальная скорость генерации получилась у сценария 'Qwen3-4B (4-bit)' чер

## Выводы

Этот ноутбук показывает несколько рабочих локальных сценариев для **macOS M2**:
- `Qwen3-4B` в более полном формате: **Transformers + MPS**, если `MPS` действительно доступен;
- `Qwen3-4B` в квантованном режиме: **MLX 4-bit**;
- `Qwen3-8B` в reasoning/thinking-сценарии: **MLX 4-bit**;
- GGUF/desktop-friendly путь: **llama.cpp + Metal**;
- API-путь для локального сервинга: **Ollama**.

Практический совет: не держи все тяжёлые рантаймы активными одновременно дольше, чем нужно. Для M2 удобнее запускать секцию, смотреть результат, освобождать память и только потом переходить к следующему backend.
